# Pinned nanochat reference baselines

This notebook exposes two explicit profiles from the same pinned upstream implementation. **d12** is the canonical CUDA/server reference. **mac_d4** is a separate Apple-MPS development baseline that preserves nanochat's architecture, initialization, Muon/AdamW grouping, and schedule logic at a feasible local scale. The Mac profile is not reported as a d12 result.

The default profile is `auto`: CUDA selects d12; MPS/CPU selects mac_d4. CUDA retains upstream `torch.compile`; MPS/CPU uses the same model and optimizer in eager mode because compile support is not part of the portable non-CUDA contract. Override with `RG_NANOCHAT_PROFILE=d12` or `RG_NANOCHAT_PROFILE=mac`. Every run is restartable from the latest complete upstream model/optimizer checkpoint.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run from a clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rg_baselines.nanochat_portable import (
    DEFAULT_NANOCHAT_SEEDS,
    NANOCHAT_COMMIT,
    analyze_weightwatcher_checkpoints,
    collect_metrics,
    detect_device_type,
    ensure_checkout,
    ensure_environment,
    prepare_data,
    resolve_profile,
    run_seed,
    summarize_training_metrics,
    summarize_weightwatcher,
)

DEVICE_TYPE = detect_device_type()
PROFILE_REQUEST = os.environ.get('RG_NANOCHAT_PROFILE', 'auto')
CONFIG = resolve_profile(PROFILE_REQUEST, device_type=DEVICE_TYPE)
SEEDS = DEFAULT_NANOCHAT_SEEDS
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3
default_nproc = torch.cuda.device_count() if DEVICE_TYPE == 'cuda' else 1
NPROC_PER_NODE = int(os.environ.get('RG_NANOCHAT_NPROC', str(max(1, default_nproc))))
if DEVICE_TYPE != 'cuda':
    NPROC_PER_NODE = 1

WORK_ROOT = Path(os.environ.get('RG_NANOCHAT_WORK_ROOT', ROOT / 'nanochat_work')).expanduser().resolve()
CHECKOUT = WORK_ROOT / 'upstream_nanochat'
CACHE = WORK_ROOT / CONFIG.profile_name / 'cache'
RUN_ROOT = Path(os.environ.get('RG_BASELINE_RUN_ROOT', ROOT / 'runs')).expanduser().resolve()
RUN_DIR = RUN_ROOT / 'nanochat' / CONFIG.profile_name
PLOT_DIR = RUN_DIR / 'plots'
for directory in (WORK_ROOT, CACHE, RUN_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('pinned commit:', NANOCHAT_COMMIT)
print('device:', DEVICE_TYPE, 'profile:', CONFIG.profile_name, 'processes:', NPROC_PER_NODE)
print('model depth/width/context:', CONFIG.depth, CONFIG.model_dim, CONFIG.max_seq_len)
display(pd.DataFrame([CONFIG.__dict__]))

## 1. Pin upstream code and create the correct environment

CUDA uses nanochat's GPU extra. Linux CPU uses its CPU index. macOS/MPS uses the native PyPI Torch wheel rather than the Linux CPU wheel. The checkout is detached at the audited commit. Two exact runtime patches are installed: the hard-coded global seed becomes configurable, and `torch.compile` becomes environment-controlled. Compilation remains enabled for canonical CUDA d12 and is disabled for MPS/CPU. No model or optimizer code is changed.

In [ ]:
CHECKOUT = ensure_checkout(CHECKOUT)
ensure_environment(CHECKOUT, device_type=DEVICE_TYPE)
print('checkout:', CHECKOUT)

## 2. Prepare the profile-specific upstream data and tokenizer

d12 retains the 1,000-shard, 2-billion-character tokenizer preparation. mac_d4 uses a separately cached 100-shard, 200-million-character preparation so local development does not overwrite or masquerade as the canonical corpus.

In [ ]:
prepare_data(CHECKOUT, CACHE, CONFIG)
print('profile cache:', CACHE)

## 3. Run or resume three complete replicates

The wrapper finds the newest checkpoint that has model, metadata, and every optimizer shard, passes `--resume-from-step`, appends to the persistent log, and writes both the upstream completion fingerprint and a versioned `runtime_policy.json`. A run created under a different device/compile policy is rejected instead of being silently reused.

In [ ]:
logs = []
for seed in SEEDS:
    log_path = run_seed(
        CHECKOUT, CACHE, RUN_DIR, CONFIG,
        seed=seed,
        device_type=DEVICE_TYPE,
        nproc_per_node=NPROC_PER_NODE,
        resume=True,
    )
    logs.append((seed, log_path))

metrics = collect_metrics(
    logs,
    RUN_DIR / 'training_metrics_all_seeds.csv',
    profile_name=CONFIG.profile_name,
)
display(metrics.groupby('seed').tail(10))

## 4. Training and validation trajectories

Only steps present for all three seeds receive a 95% Student-t interval. d12 additionally reports the native CORE score when it is evaluated; mac_d4 intentionally disables the expensive CORE suite.

In [ ]:
for metric in ['train_loss', 'validation_bpb', 'lr_multiplier', 'tokens_per_sec', 'core_metric']:
    if metric not in metrics.columns or metrics[metric].notna().sum() == 0:
        continue
    summary = summarize_training_metrics(metrics, value=metric, expected_seeds=SEEDS)
    summary.to_csv(RUN_DIR / f'{metric}_summary_95ci.csv', index=False)
    figure, axis = plt.subplots(figsize=(9, 5))
    for seed, run in metrics.dropna(subset=[metric]).groupby('seed'):
        axis.plot(run['step'], run[metric], alpha=0.18, linewidth=0.8)
    axis.plot(summary['step'], summary['mean'], linewidth=2.0, label='mean')
    axis.fill_between(summary['step'], summary['ci95_low'], summary['ci95_high'], alpha=0.16)
    axis.set_xlabel('Optimizer step')
    axis.set_ylabel(metric.replace('_', ' ').title())
    axis.set_title(f'{CONFIG.profile_name}: {metric}, 95% Student-t CI across runs')
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
    figure.tight_layout()
    figure.savefig(PLOT_DIR / f'{metric}_95ci.png', dpi=170, bbox_inches='tight')
    plt.show()

## 5. Offline hidden-matrix WeightWatcher analysis

WeightWatcher runs after training and analyzes only the six principal matrices in each transformer block. Embeddings, the unembedding, value-embedding tables, gates, and scalars are excluded. Direct `alpha`, `ERG_gap`, and randomized-MP `num_traps` values are required; no proxy or fallback is constructed.

In [ ]:
spectral_frames = []
for seed in SEEDS:
    frame = analyze_weightwatcher_checkpoints(
        CHECKOUT, CACHE,
        config=CONFIG,
        seed=seed,
        output_csv=RUN_DIR / f'weightwatcher_seed_{seed}.csv',
    )
    spectral_frames.append(frame)
spectral = pd.concat(spectral_frames, ignore_index=True)
spectral.to_csv(RUN_DIR / 'weightwatcher_all_seeds.csv', index=False)
display(spectral.groupby('seed').tail(20))

In [ ]:
matrix_colors = {
    'W_Q': '#0072B2', 'W_K': '#E69F00', 'W_V': '#009E73',
    'W_O': '#D55E00', 'W_MLP_IN': '#CC79A7', 'W_MLP_OUT': '#56B4E9',
}
for metric in ['alpha', 'ERG_gap', 'num_traps']:
    summary = summarize_weightwatcher(spectral, metric, expected_seeds=SEEDS)
    summary.to_csv(RUN_DIR / f'{metric}_by_matrix_95ci.csv', index=False)
    assert summary['n'].eq(3).all()
    for block, block_frame in summary.groupby('block'):
        figure, axis = plt.subplots(figsize=(10, 5.5))
        for matrix_type, curve in block_frame.groupby('matrix_type'):
            curve = curve.sort_values('step')
            color = matrix_colors[matrix_type]
            axis.plot(curve['step'], curve['mean'], color=color, linewidth=2.0, label=matrix_type)
            axis.fill_between(curve['step'], curve['ci95_low'], curve['ci95_high'], color=color, alpha=0.14)
        if metric == 'alpha':
            axis.axhline(2.0, color='black', linestyle='--', linewidth=1.0)
        if metric == 'ERG_gap':
            axis.axhline(0.0, color='black', linestyle='--', linewidth=1.0)
        axis.set_xlabel('Checkpoint step')
        axis.set_ylabel(metric)
        axis.set_title(f'{CONFIG.profile_name} block {int(block)}: {metric}')
        axis.grid(alpha=0.25)
        axis.legend(frameon=False, ncol=2)
        figure.tight_layout()
        figure.savefig(PLOT_DIR / f'block_{int(block):02d}_{metric}.png', dpi=170, bbox_inches='tight')
        plt.show()

## Baseline contract

Do not silently update the pinned nanochat commit. A different upstream commit, profile, process count, device/compile policy, or data/tokenizer preparation creates a new baseline version. d12 and mac_d4 results must remain separately labeled.